In [6]:
# Cell 1: imports and dataset loading
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mutual_info_score

df = pd.read_csv("/home/abdoulaye/Documents/Orange Digital center/Machine Learning Zoomcamp 2026/course_lead_scoring_2026.csv")
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [7]:
# Cell 2: check missing values
print(df.isnull().sum())

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64


In [9]:
# Cell 3: fill missing values
# categorical -> 'NA', numerical -> 0.0
categorical = df.select_dtypes(include='object').columns.tolist()
numerical = df.select_dtypes(exclude='object').columns.tolist()

# converted is the target, remove it from numerical features list if present
if 'converted' in numerical:
    numerical.remove('converted')

print("Categorical:", categorical)
print("Numerical:", numerical)

for col in categorical:
    df[col] = df[col].fillna('NA')
for col in numerical:
    df[col] = df[col].fillna(0.0)

Categorical: ['lead_source', 'industry', 'employment_status', 'location']
Numerical: ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


/tmp/ipykernel_108627/754213711.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical = df.select_dtypes(include='object').columns.tolist()


In [ ]:
#Cell4
df['industry'].mode()[0]

'technology'

In [ ]:
# Cell 5
df[numerical].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [ ]:
# Cell6
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

del df_train['converted']
del df_val['converted']
del df_test['converted']

print(len(df_train), len(df_val), len(df_test))

3000 1000 1000


In [ ]:
# Cell 7
def mutual_info_converted_score(series):
    return round(mutual_info_score(series, y_train), 2)

mi_scores = df_train[categorical].apply(mutual_info_converted_score)
mi_scores.sort_values(ascending=False)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

In [ ]:
# Cell 8
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
accuracy = (y_pred == y_val).mean()
round(accuracy, 2)

np.float64(0.64)

In [ ]:
# Cell 9
for feature in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
    subset = [f for f in all_features if f != feature]
    print(feature, "| difference:", original_accuracy - train_and_score(subset))

lead_source | difference: 0.0030000000000000027
number_of_courses_viewed | difference: 0.0020000000000000018
interaction_count | difference: 0.04400000000000004


In [ ]:
# Cell 10
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    print(C, round(train_and_score(all_features, C=C), 3))

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645
